# L14 · Árboles y bosques causales

**S14 · Lectura: Cap. 13 · 40–55 min**

## La pregunta

¿La heterogeneidad es señal o ruido?

## Qué harás

Entender honestidad, partición y estabilidad en bosques causales.

- Lee la idea breve.
- Ejecuta la demostración y mira el gráfico.
- Completa las celdas marcadas.
- Escribe qué significa el resultado y qué no permite afirmar.

**Producto:** Estimaciones por grupo y pruebas de estabilidad.

> **Idea clave:** Separar dónde se busca una división de dónde se estima el efecto reduce el sobreajuste.

La solución está en `la ruta de rescate docente (no incluida)`. Intenta primero el notebook del estudiante.

In [ ]:
# Localiza lab_common.py (carpeta actual o una superior)
import sys, pathlib
for _p in [pathlib.Path.cwd(), *list(pathlib.Path.cwd().parents)[:3]]:
    if (_p / "lab_common.py").exists():
        sys.path.insert(0, str(_p)); break

## 0. Preparación

In [ ]:
from lab_common import *
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
rng = setup()
N_REP = R(30, 300)
B_ARB = R(40, 200)
print("réplicas:", N_REP, "· árboles del bosque:", B_ARB)

## 1. Honestidad (Def. 13.1, §13.1)
Un árbol que **elige** los cortes con unos datos y **estima** el efecto de cada hoja con los mismos datos se engaña: los cortes se eligieron justo donde el ruido favorecía una diferencia. La receta honesta:

1. Partir la muestra en $S_{\text{split}}$ y $S_{\text{est}}$.
2. Construir el árbol (los cortes) con $S_{\text{split}}$.
3. Estimar el efecto de cada hoja con $S_{\text{est}}$, con su error estándar.

Para la variable que el árbol predice usamos el **score doblemente robusto** (AIPW) con nuisance ajustados por *cross-fitting* en dos mitades, de modo que la media de ese score dentro de una hoja estima el efecto de la hoja aun con confusión observada.

In [ ]:
def make_dgp(n=2000, p=10, seed=0, het=1.0):
    r = np.random.default_rng(seed); X = r.uniform(-1, 1, (n, p)); e = 1 / (1 + np.exp(-1.0 - 0.5 * X[:, 0]))
    D = (r.uniform(size=n) < e).astype(int); tau = 2 + het * (3 * X[:, 0] - X[:, 1] ** 2); mu0 = X[:, 2] + .5 * X[:, 3]
    Y = mu0 + D * tau + r.standard_normal(n) * .5
    return Y, D, X, tau

ETR = lambda: ExtraTreesRegressor(n_estimators=40, min_samples_leaf=5, max_features=0.8, random_state=0)
def psi_cf(X, y, d, K=2, seed=SEED):
    """Score AIPW por unidad con nuisance de otros pliegues (cross-fitting)."""
    n = len(y); psi = np.zeros(n)
    for te in kfold(n, K, seed):
        tr = np.setdiff1d(np.arange(n), te); t1, t0 = tr[d[tr] == 1], tr[d[tr] == 0]
        m1 = ETR().fit(X[t1], y[t1]).predict(X[te]); m0 = ETR().fit(X[t0], y[t0]).predict(X[te])
        e = np.clip(LogisticRegression(max_iter=500).fit(X[tr], d[tr]).predict_proba(X[te])[:, 1], .02, .98)
        psi[te] = m1 - m0 + d[te] * (y[te] - m1) / e - (1 - d[te]) * (y[te] - m0) / (1 - e)
    return psi

Y, D, X, tau = make_dgp(2000, seed=42)
psi = psi_cf(X, Y, D)
print("media del score AIPW:", psi.mean().round(3), "· τ medio verdadero:", tau.mean().round(3))

### Completa tú: la estimación honesta de las hojas
`hojas_honestas(arbol, X_est, psi_est)` recibe un árbol ya construido (sklearn) y la muestra de **estimación**. Debe devolver, **por hoja**, `n`, `tau` (media de `psi_est` en la hoja), `se` ($=\mathrm{sd}/\sqrt n$) y el IC 95 %. Pista: `arbol.apply(X)` da el índice de hoja de cada fila.

In [ ]:
# ✏️ Completa tú — hojas_honestas
# Pista: Para cada hoja h: v = psi_est[hoja == h]; tau = v.mean(); se = v.std(ddof=1)/sqrt(len(v)).
def hojas_honestas(arbol, X_est, psi_est):
    """DataFrame indexado por hoja con columnas n, tau, se, lo, hi (usando SOLO la muestra de estimación)."""
    hoja = arbol.apply(X_est)
    raise NotImplementedError("completa hojas_honestas")

## Sección de evaluación reservada

La edición pública termina antes de las celdas docentes que comparan con verdad simulada, efectos potenciales o resultados oráculo. Esa revisión se ejecuta aparte; aquí debes sostener la decisión con observables, incertidumbre y supuestos declarados.
